# Dual Analysis of the LP Relaxation

In [42]:
import pandas as pd
import numpy as np
import pulp

# ================================
# 1) Baseline data (same as sens_anal.ipynb)
# ================================
customers = [f"C{i+1}" for i in range(10)]  # m = 10
locations = [f"T{j+1}" for j in range(8)]   # n = 8

r = 10.0   # revenue per unit
k = 5.0    # cost per unit (ingredient)
f = 250.0  # fixed cost per truck per day

demands = {
    "C1": 65, "C2": 40, "C3": 55, "C4": 30, "C5": 75,
    "C6": 50, "C7": 60, "C8": 35, "C9": 45, "C10": 70,
}

alpha = {
    "C1":  {"T1": 0.9, "T2": 0.7, "T3": 0.2, "T4": 0.1, "T5": 0.0, "T6": 0.3, "T7": 0.6, "T8": 0.4},
    "C2":  {"T1": 0.4, "T2": 0.8, "T3": 0.1, "T4": 0.0, "T5": 0.6, "T6": 0.2, "T7": 0.3, "T8": 0.7},
    "C3":  {"T1": 0.2, "T2": 0.3, "T3": 0.9, "T4": 0.5, "T5": 0.1, "T6": 0.0, "T7": 0.6, "T8": 0.4},
    "C4":  {"T1": 0.1, "T2": 0.0, "T3": 0.4, "T4": 0.8, "T5": 0.9, "T6": 0.3, "T7": 0.5, "T8": 0.2},
    "C5":  {"T1": 0.7, "T2": 0.6, "T3": 0.3, "T4": 0.0, "T5": 0.4, "T6": 0.9, "T7": 0.1, "T8": 0.5},
    "C6":  {"T1": 0.5, "T2": 0.2, "T3": 0.1, "T4": 0.3, "T5": 0.6, "T6": 0.4, "T7": 0.8, "T8": 0.7},
    "C7":  {"T1": 0.3, "T2": 0.4, "T3": 0.0, "T4": 0.6, "T5": 0.2, "T6": 0.1, "T7": 0.9, "T8": 0.8},
    "C8":  {"T1": 0.6, "T2": 0.5, "T3": 0.7, "T4": 0.1, "T5": 0.3, "T6": 0.0, "T7": 0.2, "T8": 0.9},
    "C9":  {"T1": 0.2, "T2": 0.1, "T3": 0.6, "T4": 0.9, "T5": 0.4, "T6": 0.8, "T7": 0.0, "T8": 0.3},
    "C10": {"T1": 0.8, "T2": 0.3, "T3": 0.5, "T4": 0.2, "T5": 0.7, "T6": 0.1, "T7": 0.4, "T8": 0.6},
}

# Compute beta_ij = (r-k) * alpha_ij * d_i
beta = {}
for i in customers:
    for j in locations:
        beta[(i, j)] = (r - k) * alpha[i][j] * demands[i]

print(f"Customers (m): {len(customers)}, Locations (n): {len(locations)}")
print(f"r={r}, k={k}, f={f}")
print(f"Nonzero beta entries: {sum(1 for v in beta.values() if v > 0)}")
print("\nSample beta values:")
for i in ["C1", "C5", "C10"]:
    for j in ["T1", "T4", "T8"]:
        print(f"  beta[{i},{j}] = {beta[(i,j)]:6.1f}  (alpha={alpha[i][j]}, d={demands[i]})")

Customers (m): 10, Locations (n): 8
r=10.0, k=5.0, f=250.0
Nonzero beta entries: 72

Sample beta values:
  beta[C1,T1] =  292.5  (alpha=0.9, d=65)
  beta[C1,T4] =   32.5  (alpha=0.1, d=65)
  beta[C1,T8] =  130.0  (alpha=0.4, d=65)
  beta[C5,T1] =  262.5  (alpha=0.7, d=75)
  beta[C5,T4] =    0.0  (alpha=0.0, d=75)
  beta[C5,T8] =  187.5  (alpha=0.5, d=75)
  beta[C10,T1] =  280.0  (alpha=0.8, d=70)
  beta[C10,T4] =   70.0  (alpha=0.2, d=70)
  beta[C10,T8] =  210.0  (alpha=0.6, d=70)


In [43]:
# ================================
# 2) Build the LP relaxation model
# ================================
# Same structure as IP but with variables in [0, 1]

prob = pulp.LpProblem("Dumplings_LP_Relaxation", pulp.LpMaximize)

# Decision variables (continuous, bounds [0,1])
x = pulp.LpVariable.dicts("x", locations, lowBound=0, upBound=1, cat=pulp.LpContinuous)
y = pulp.LpVariable.dicts("y", (customers, locations), lowBound=0, upBound=1, cat=pulp.LpContinuous)

# Objective: maximize profit
prob += (
    pulp.lpSum((r - k) * alpha[i][j] * demands[i] * y[i][j] for i in customers for j in locations)
    - pulp.lpSum(f * x[j] for j in locations)
), "Profit"

# Constraint 1: each customer served at most once — dual variable u_i
forall_i_constraints = {}  # store constraint objects for dual extraction
for i in customers:
    c_name = f"UniqueServe_{i}"
    forall_i_constraints[i] = prob.addConstraint(
        pulp.lpSum(y[i][j] for j in locations) <= 1,
        name=c_name
    )

# Constraint 2: can only assign if truck is open — dual variable v_{ij}
link_constraints = {}  # store constraint objects for dual extraction
for i in customers:
    for j in locations:
        c_name = f"LinkTruck_{i}_{j}"
        link_constraints[(i, j)] = prob.addConstraint(
            y[i][j] - x[j] <= 0,
            name=c_name
        )

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"=== LP Relaxation Solved ===")
print(f"Status: {pulp.LpStatus[prob.status]}")
print(f"LP Optimal Profit: {pulp.value(prob.objective):.4f}")

# Compare with IP result (baseline: 928.0)
ip_profit = 928.0
print(f"IP Optimal Profit (baseline): {ip_profit}")
print(f"Integrality Gap: {abs(pulp.value(prob.objective) - ip_profit):.4f}")

# Print primal solution
print("\n--- Primal LP Solution ---")
print(f"Opened trucks (x_j > 0):")
for j in locations:
    x_val = pulp.value(x[j])
    if x_val > 1e-6:
        print(f"  x[{j}] = {x_val:.4f}")

print(f"\nAssignment (y_ij > 0):")
for i in customers:
    for j in locations:
        y_val = pulp.value(y[i][j])
        if y_val > 1e-6:
            print(f"  y[{i},{j}] = {y_val:.4f}  (beta={beta[(i,j)]:.1f})")

=== LP Relaxation Solved ===
Status: Optimal
LP Optimal Profit: 1293.7500
IP Optimal Profit (baseline): 928.0
Integrality Gap: 365.7500

--- Primal LP Solution ---
Opened trucks (x_j > 0):
  x[T1] = 0.5000
  x[T6] = 0.5000
  x[T7] = 0.5000
  x[T8] = 0.5000

Assignment (y_ij > 0):
  y[C1,T1] = 0.5000  (beta=292.5)
  y[C1,T7] = 0.5000  (beta=195.0)
  y[C2,T1] = 0.5000  (beta=80.0)
  y[C2,T8] = 0.5000  (beta=140.0)
  y[C3,T7] = 0.5000  (beta=165.0)
  y[C3,T8] = 0.5000  (beta=110.0)
  y[C4,T6] = 0.5000  (beta=45.0)
  y[C4,T7] = 0.5000  (beta=75.0)
  y[C5,T1] = 0.5000  (beta=262.5)
  y[C5,T6] = 0.5000  (beta=337.5)
  y[C6,T7] = 0.5000  (beta=200.0)
  y[C6,T8] = 0.5000  (beta=175.0)
  y[C7,T7] = 0.5000  (beta=270.0)
  y[C7,T8] = 0.5000  (beta=240.0)
  y[C8,T1] = 0.5000  (beta=105.0)
  y[C8,T8] = 0.5000  (beta=157.5)
  y[C9,T6] = 0.5000  (beta=180.0)
  y[C9,T8] = 0.5000  (beta=67.5)
  y[C10,T1] = 0.5000  (beta=280.0)
  y[C10,T8] = 0.5000  (beta=210.0)


In [44]:
# ================================
# 3) Extract dual variables (shadow prices) from LP solver
# ================================
# In PuLP, constraint.pi gives the dual value (shadow price) of the constraint.

# Dual variable u_i (for UniqueServe_i constraint)
u_opt = {}
for i in customers:
    u_opt[i] = prob.constraints[f"UniqueServe_{i}"].pi
    if u_opt[i] is None:
        u_opt[i] = 0.0
    else:
        u_opt[i] = float(u_opt[i])

# Dual variable v_{ij} (for LinkTruck_i_j constraint)
v_opt = {}
for i in customers:
    for j in locations:
        v_opt[(i, j)] = prob.constraints[f"LinkTruck_{i}_{j}"].pi
        if v_opt[(i, j)] is None:
            v_opt[(i, j)] = 0.0
        else:
            v_opt[(i, j)] = float(v_opt[(i, j)])

print("=== Optimal Dual Variables ===\n")

print("--- u_i (shadow price of 'customer i served at most once' constraint) ---")
u_df = pd.DataFrame([
    {"Customer": i, "demand": demands[i],
     "u_i (Shadow Price)": u_opt[i]}
    for i in customers
]).sort_values("u_i (Shadow Price)", ascending=False)
print(u_df.to_string(index=False))

print("\n--- v_{ij} (shadow price of 'y_ij <= x_j' constraint) — nonzero only ---")
v_nonzero = [
    {"Customer": i, "Location": j,
     "beta_ij": beta[(i, j)],
     "v_{ij} (Shadow Price)": v_opt[(i, j)]}
    for (i, j) in v_opt if abs(v_opt[(i, j)]) > 1e-6
]
if v_nonzero:
    v_df = pd.DataFrame(v_nonzero).sort_values("v_{ij} (Shadow Price)", ascending=False)
    print(v_df.to_string(index=False))
else:
    print("  (all v_ij = 0)")

=== Optimal Dual Variables ===

--- u_i (shadow price of 'customer i served at most once' constraint) ---
Customer  demand  u_i (Shadow Price)
      C5      75              208.75
      C1      65              195.00
      C7      60              185.00
     C10      70              181.25
      C6      50              125.00
      C3      55              110.00
      C8      35              105.00
      C2      40               80.00
      C9      45               63.75
      C4      30               40.00

--- v_{ij} (shadow price of 'y_ij <= x_j' constraint) — nonzero only ---
Customer Location  beta_ij  v_{ij} (Shadow Price)
      C9       T4    202.5                 138.75
      C3       T3    247.5                 137.50
      C5       T6    337.5                 128.75
      C9       T6    180.0                 116.25
     C10       T1    280.0                  98.75
      C1       T1    292.5                  97.50
      C4       T5    135.0                  95.00
      C7     

In [45]:
# ================================
# 4) Verify strong duality: compute dual objective
# ================================
# Dual objective:
#   min Σ_i u_i + Σ_{i,j} max{0, β_ij - (u_i+v_ij)} + Σ_j max{0, -f + Σ_i v_ij}

primal_obj = pulp.value(prob.objective)

# Term 1: Σ_i u_i
term1 = sum(u_opt[i] for i in customers)

# Term 2: Σ_{i,j} max{0, β_ij - (u_i + v_ij)}
term2 = sum(
    max(0.0, beta[(i, j)] - (u_opt[i] + v_opt[(i, j)]))
    for i in customers for j in locations
)

# Term 3: Σ_j max{0, -f + Σ_i v_ij}
term3 = sum(
    max(0.0, -f + sum(v_opt[(i, j)] for i in customers))
    for j in locations
)

dual_obj = term1 + term2 + term3

print("=== Strong Duality Verification ===")
print(f"Primal Optimal Value (LP):  {primal_obj:12.4f}")
print(f"Dual Objective Value:       {dual_obj:12.4f}")
print(f"Difference (should be ~0):   {abs(primal_obj - dual_obj):10.6f}")
print()
print(f"Term breakdown:")
print(f"  Σ_i u_i                = {term1:10.4f}")
print(f"  Σ_{i,j} max(0, β-(u+v)) = {term2:10.4f}")
print(f"  Σ_j max(0, -f+Σ_i v)   = {term3:10.4f}")
print(f"  Total dual objective   = {dual_obj:10.4f}")

# Also compute λ* values for verification
# λ*_y for each (i,j): max{0, beta_ij - (u_i + v_ij)}
# λ*_x for each j: max{0, -f + Σ_i v_ij}
lambda_y = {(i, j): max(0.0, beta[(i, j)] - (u_opt[i] + v_opt[(i, j)]))
            for i in customers for j in locations}
lambda_x = {j: max(0.0, -f + sum(v_opt[(i, j)] for i in customers))
            for j in locations}

print(f"\nλ* (upper-bound duals) — nonzero only:")
lambda_y_nonzero = [k for k, v in lambda_y.items() if v > 1e-6]
lambda_x_nonzero = [k for k, v in lambda_x.items() if v > 1e-6]
print(f"  λ_y (y_ij <= 1): {len(lambda_y_nonzero)} nonzero out of {len(lambda_y)}")
print(f"  λ_x (x_j  <= 1): {len(lambda_x_nonzero)} nonzero out of {len(lambda_x)}")
for k in lambda_y_nonzero:
    print(f"    λ_y[{k[0]},{k[1]}] = {lambda_y[k]:.4f}")
for k in lambda_x_nonzero:
    print(f"    λ_x[{k}] = {lambda_x[k]:.4f}")

=== Strong Duality Verification ===
Primal Optimal Value (LP):     1293.7500
Dual Objective Value:          1293.7500
Difference (should be ~0):     0.000000

Term breakdown:
  Σ_i u_i                =  1293.7500
  Σ_('C10', 'T8') max(0, β-(u+v)) =     0.0000
  Σ_j max(0, -f+Σ_i v)   =     0.0000
  Total dual objective   =  1293.7500

λ* (upper-bound duals) — nonzero only:
  λ_y (y_ij <= 1): 0 nonzero out of 80
  λ_x (x_j  <= 1): 0 nonzero out of 8


In [46]:
# ================================
# 6) Sensitivity: How do dual variables change with f?
# ================================
# We re-solve the LP for a range of f values and track dual variables.

print("=== Sensitivity of Dual Variables to Fixed Cost f ===\n")

f_range = np.arange(150, 401, 25)
dual_sensitivity = []

for f_test in f_range:
    # Build a fresh LP for each f
    prob_tmp = pulp.LpProblem("Dumplings_LP_Relaxation_Tmp", pulp.LpMaximize)
    x_tmp = pulp.LpVariable.dicts("x", locations, lowBound=0, upBound=1, cat=pulp.LpContinuous)
    y_tmp = pulp.LpVariable.dicts("y", (customers, locations), lowBound=0, upBound=1, cat=pulp.LpContinuous)
    
    prob_tmp += (
        pulp.lpSum((r - k) * alpha[i][j] * demands[i] * y_tmp[i][j] for i in customers for j in locations)
        - pulp.lpSum(f_test * x_tmp[j] for j in locations)
    ), "Profit"
    
    for i in customers:
        prob_tmp += pulp.lpSum(y_tmp[i][j] for j in locations) <= 1, f"US_{i}"
    for i in customers:
        for j in locations:
            prob_tmp += y_tmp[i][j] - x_tmp[j] <= 0, f"LT_{i}_{j}"
    
    prob_tmp.solve(pulp.PULP_CBC_CMD(msg=0))
    
    primal_val = pulp.value(prob_tmp.objective)
    
    # Extract u_i
    u_vals = {i: float(prob_tmp.constraints[f"US_{i}"].pi or 0.0) for i in customers}
    
    # Count nonzero u_i and sum of u_i
    n_u_nonzero = sum(1 for v in u_vals.values() if abs(v) > 1e-6)
    sum_u = sum(u_vals.values())
    
    # Open trucks (use epsilon to catch LP fractional values like 0.5)
    open_trucks = [j for j in locations if pulp.value(x_tmp[j]) > 1e-6]
    n_open = len(open_trucks)
    
    dual_sensitivity.append({
        "f": float(f_test),
        "LP_Profit": primal_val,
        "Sum_u_i": sum_u,
        "N_u_i_Nonzero": n_u_nonzero,
        "N_Open_Trucks": n_open,
        "Open_Trucks": ", ".join(open_trucks),
    })

df_dual_sens = pd.DataFrame(dual_sensitivity)
print(df_dual_sens.to_string(index=False))


=== Sensitivity of Dual Variables to Fixed Cost f ===

    f  LP_Profit  Sum_u_i  N_u_i_Nonzero  N_Open_Trucks        Open_Trucks
150.0    1557.50  1557.50             10              3         T1, T4, T8
175.0    1482.50  1482.50             10              3         T1, T4, T8
200.0    1412.50  1412.50             10              5 T1, T4, T6, T7, T8
225.0    1350.00  1350.00             10              5 T1, T4, T6, T7, T8
250.0    1293.75  1293.75             10              4     T1, T6, T7, T8
275.0    1243.75  1243.75             10              4     T1, T6, T7, T8
300.0    1193.75  1193.75             10              4     T1, T6, T7, T8
325.0    1156.25  1156.25             10              3         T1, T7, T8
350.0    1118.75  1118.75             10              3         T1, T7, T8
375.0    1081.25  1081.25             10              3         T1, T7, T8
400.0    1047.50  1047.50             10              1                 T8


In [47]:
rows = []
for i in customers:
    y_served = [j for j in locations if pulp.value(y[i][j]) > 1e-6]
    total_y = sum(pulp.value(y[i][j]) for j in locations)
    rows.append({
        "Customer": i,
        "Demand": demands[i],
        "u_i": round(u_opt[i], 4),
        "Total_y_ij": round(total_y, 4),
        "Served_From": ", ".join(y_served) if y_served else "unserved",
    })
print(pd.DataFrame(rows).sort_values("u_i", ascending=False).to_string(index=False))

v_rows = []
for i in customers:
    for j in locations:
        if abs(v_opt[(i, j)]) > 1e-6:
            v_rows.append({
                "Pair": f"({i},{j})",
                "beta_ij": round(beta[(i, j)], 1),
                "v_ij": round(v_opt[(i, j)], 4),
                "y_ij": round(pulp.value(y[i][j]), 4),
                "u_i_plus_v_ij": round(u_opt[i] + v_opt[(i, j)], 4),
            })
if v_rows:
    print(pd.DataFrame(v_rows).sort_values("v_ij", ascending=False).to_string(index=False))
else:
    print("All v_ij = 0")

cs_rows = []
for i in customers:
    sum_y = sum(pulp.value(y[i][j]) for j in locations)
    cs_rows.append({
        "Customer": i,
        "u_i_slack": round(u_opt[i] * (1 - sum_y), 6),
    })
for i in customers:
    for j in locations:
        cs_rows.append({
            "Pair": f"({i},{j})",
            "v_slack": round(v_opt[(i, j)] * (pulp.value(x[j]) - pulp.value(y[i][j])), 6),
        })
print(pd.DataFrame(cs_rows).to_string(index=False))

Customer  Demand    u_i  Total_y_ij Served_From
      C5      75 208.75         1.0      T1, T6
      C1      65 195.00         1.0      T1, T7
      C7      60 185.00         1.0      T7, T8
     C10      70 181.25         1.0      T1, T8
      C6      50 125.00         1.0      T7, T8
      C3      55 110.00         1.0      T7, T8
      C8      35 105.00         1.0      T1, T8
      C2      40  80.00         1.0      T1, T8
      C9      45  63.75         1.0      T6, T8
      C4      30  40.00         1.0      T6, T7
    Pair  beta_ij   v_ij  y_ij  u_i_plus_v_ij
 (C9,T4)    202.5 138.75   0.0          202.5
 (C3,T3)    247.5 137.50   0.0          247.5
 (C5,T6)    337.5 128.75   0.5          337.5
 (C9,T6)    180.0 116.25   0.5          180.0
(C10,T1)    280.0  98.75   0.5          280.0
 (C1,T1)    292.5  97.50   0.5          292.5
 (C4,T5)    135.0  95.00   0.0          135.0
 (C7,T7)    270.0  85.00   0.5          270.0
 (C4,T4)    120.0  80.00   0.0          120.0
 (C2,T2)    